# Selecta Claims Analysis

This notebook processes and analyses claims data for the quarterly valuation process.

**Steps:**
1. Load claims data
2. Clean and validate data
3. Summarise claims by category
4. Calculate incurred claims (paid + IBNR reserve)
5. Export results for use in the quarterly valuation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## 1. Load Claims Data

In [ ]:
# Load claims data
# claims_df = pd.read_csv('data/claims.csv')
# Placeholder: create sample claims data
claims_df = pd.DataFrame({
    'claim_id': range(1, 101),
    'policy_id': np.random.randint(1000, 2000, 100),
    'claim_date': pd.date_range('2024-01-01', periods=100, freq='3D'),
    'claim_type': np.random.choice(['Death', 'Disability', 'Critical Illness'], 100),
    'paid_amount': np.random.uniform(1000, 100000, 100).round(2),
    'ibnr_reserve': np.random.uniform(0, 20000, 100).round(2),
    'status': np.random.choice(['Open', 'Closed', 'Pending'], 100),
})

print(f'Claims loaded: {len(claims_df)} rows')
claims_df.head()

## 2. Clean and Validate Data

In [ ]:
# Check for nulls
print('Null counts:')
print(claims_df.isnull().sum())

# Check claim amounts are non-negative
assert (claims_df['paid_amount'] >= 0).all(), 'Negative paid amounts found'
assert (claims_df['ibnr_reserve'] >= 0).all(), 'Negative IBNR reserves found'

print('\nData validation passed.')

## 3. Summarise Claims by Category

In [ ]:
summary = claims_df.groupby('claim_type').agg(
    count=('claim_id', 'count'),
    total_paid=('paid_amount', 'sum'),
    total_ibnr=('ibnr_reserve', 'sum'),
).reset_index()

summary['total_incurred'] = summary['total_paid'] + summary['total_ibnr']
print(summary.to_string(index=False))

## 4. Calculate Incurred Claims

In [ ]:
claims_df['incurred'] = claims_df['paid_amount'] + claims_df['ibnr_reserve']

total_paid = claims_df['paid_amount'].sum()
total_ibnr = claims_df['ibnr_reserve'].sum()
total_incurred = claims_df['incurred'].sum()

print(f'Total Paid:     £{total_paid:,.2f}')
print(f'Total IBNR:     £{total_ibnr:,.2f}')
print(f'Total Incurred: £{total_incurred:,.2f}')

## 5. Visualise Claims

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Claims count by type
summary.plot.bar(x='claim_type', y='count', ax=axes[0], legend=False, color='steelblue')
axes[0].set_title('Number of Claims by Type')
axes[0].set_xlabel('Claim Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Incurred claims by type
summary.plot.bar(x='claim_type', y='total_incurred', ax=axes[1], legend=False, color='coral')
axes[1].set_title('Total Incurred Claims by Type')
axes[1].set_xlabel('Claim Type')
axes[1].set_ylabel('Amount (£)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 6. Export Results

In [ ]:
# os.makedirs('output', exist_ok=True)
# summary.to_csv('output/claims_summary.csv', index=False)
# claims_df.to_csv('output/claims_detail.csv', index=False)
print('Claims analysis complete. Results ready for quarterly valuation.')